# Xarray-Spatial Emerging Hotspots: Gi*, Mann-Kendall, and trend classification

A single hotspot map shows where clusters are right now, but not whether they're growing, fading, or brand new. `emerging_hotspots` runs Getis-Ord Gi* at each time step and applies a Mann-Kendall trend test on each pixel's time series, classifying every cell into one of 17 trend categories.

### What you'll build

1. Build a synthetic space-time cube with planted hot and cold spot signals
2. Run `emerging_hotspots` and map the 17 trend categories
3. Inspect Mann-Kendall trend z-scores and p-values
4. Drill into Gi* time series at individual pixels
5. Watch confidence bins evolve across time steps

![Emerging hotspot category map](images/emerging_hotspots_preview.png)

[Synthetic space-time cube](#Synthetic-space-time-cube) · [Category map](#Category-map) · [Mann-Kendall trend map](#Mann-Kendall-trend-map) · [Time series at individual pixels](#Time-series-at-individual-pixels) · [Confidence bins over time](#Confidence-bins-over-time)

Standard imports plus `emerging_hotspots` from xrspatial.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

from xrspatial.emerging_hotspots import emerging_hotspots

## Synthetic space-time cube

`emerging_hotspots` takes a 3D raster with dimensions `(time, y, x)`. To exercise the classifier, we plant six signals on top of random noise in a 20-step, 100×100 grid:

| Region | Signal | Expected category |
|--------|--------|-------------------|
| Centre-left | Strong positive at every step | Persistent Hot (4) |
| Centre-right | Strong negative at every step | Persistent Cold (−4) |
| Top-left | Positive only at the final step | New Hot (1) |
| Top-right | Positive ramp, weak to strong | Intensifying Hot (3) |
| Bottom-left | Positive ramp, strong to weak | Diminishing Hot (5) |
| Bottom-right | Positive at every step except the last | Historical Hot (8) |

In [ ]:
n_times, ny, nx = 20, 100, 100
rng = np.random.default_rng(42)

data = rng.standard_normal((n_times, ny, nx)).astype(np.float32)

# Persistent hot spot (centre-left)
data[:, 40:50, 20:30] += 30.0

# Persistent cold spot (centre-right)
data[:, 40:50, 70:80] -= 30.0

# New hot spot (top-left): signal only at the final step
data[-1, 10:18, 10:18] += 40.0

# Intensifying hot spot (top-right): ramp from weak to strong
for t in range(n_times):
    strength = 5.0 + 30.0 * (t / (n_times - 1))
    data[t, 10:18, 75:83] += strength

# Diminishing hot spot (bottom-left): ramp from strong to weak
for t in range(n_times):
    strength = 35.0 - 30.0 * (t / (n_times - 1))
    data[t, 80:88, 10:18] += strength

# Historical hot spot (bottom-right): strong everywhere except last
data[:-1, 80:88, 75:83] += 30.0

raster = xr.DataArray(data, dims=['time', 'y', 'x'])
print(raster)

The planted regions are small 8×8 or 10×10 blocks. Everything else is standard-normal noise.

In [ ]:
vmin, vmax = float(np.nanpercentile(data, 2)), float(np.nanpercentile(data, 98))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

raster.isel(time=0).plot.imshow(ax=ax1, cmap='RdBu_r', vmin=vmin, vmax=vmax,
                                add_colorbar=False)
ax1.set_title('First time step (t=0)', fontsize=13)
ax1.set_axis_off()

raster.isel(time=-1).plot.imshow(ax=ax2, cmap='RdBu_r', vmin=vmin, vmax=vmax,
                                 add_colorbar=True,
                                 cbar_kwargs={'label': 'Value', 'shrink': 0.7})
ax2.set_title('Last time step (t=19)', fontsize=13)
ax2.set_axis_off()

plt.tight_layout()

## Category map

`emerging_hotspots` takes the 3D raster and a 2D spatial kernel, runs Gi* at each time step, then classifies each pixel's trend. The hot-spot categories are:

| Code | Name | Rule |
|:----:|------|------|
| 1 | New | Hot only at the final step |
| 2 | Consecutive | Hot for the last N consecutive steps (N ≥ 2), never before |
| 3 | Intensifying | Hot ≥ 90% of steps incl. the last, Mann-Kendall trend up |
| 4 | Persistent | Hot ≥ 90% of steps incl. the last, no significant trend |
| 5 | Diminishing | Hot ≥ 90% of steps incl. the last, Mann-Kendall trend down |
| 6 | Sporadic | Hot at the final step, < 90% of steps, never cold |
| 7 | Oscillating | Hot at the final step, was cold at least once |
| 8 | Historical | Hot ≥ 90% of steps, but not at the final step |

Codes −1 through −8 mirror these for cold spots. Code 0 means no significant pattern.

We use a 5×5 uniform kernel. The kernel size controls what spatial scale of clustering Gi* picks up.

In [ ]:
kernel = np.ones((5, 5), dtype=np.float32)
result = emerging_hotspots(raster, kernel)

# Color scheme: warm for hot, cool for cold, gray for no pattern
CATEGORY_INFO = [
    (-8, 'Historical Cold',   '#08306b'),
    (-7, 'Oscillating Cold',  '#08519c'),
    (-6, 'Sporadic Cold',     '#2171b5'),
    (-5, 'Diminishing Cold',  '#4292c6'),
    (-4, 'Persistent Cold',   '#6baed6'),
    (-3, 'Intensifying Cold', '#9ecae1'),
    (-2, 'Consecutive Cold',  '#c6dbef'),
    (-1, 'New Cold',          '#deebf7'),
    ( 0, 'No Pattern',        '#d9d9d9'),
    ( 1, 'New Hot',           '#fee0d2'),
    ( 2, 'Consecutive Hot',   '#fcbba1'),
    ( 3, 'Intensifying Hot',  '#fc9272'),
    ( 4, 'Persistent Hot',    '#fb6a4a'),
    ( 5, 'Diminishing Hot',   '#ef3b2c'),
    ( 6, 'Sporadic Hot',      '#cb181d'),
    ( 7, 'Oscillating Hot',   '#a50f15'),
    ( 8, 'Historical Hot',    '#67000d'),
]

codes  = [c for c, _, _ in CATEGORY_INFO]
colors = [c for _, _, c in CATEGORY_INFO]

cmap = ListedColormap(colors)
boundaries = [c - 0.5 for c in codes] + [codes[-1] + 0.5]
norm = BoundaryNorm(boundaries, cmap.N)

cat = result['category']

fig, ax = plt.subplots(figsize=(10, 7.5))
cat.plot.imshow(ax=ax, cmap=cmap, norm=norm, add_colorbar=False,
                interpolation='nearest')

unique_cats = sorted(set(int(v) for v in np.unique(cat.values) if np.isfinite(v)))
ax.legend(
    handles=[Patch(facecolor=clr, edgecolor='gray', label=f'{code:+d}  {lbl}')
             for code, lbl, clr in CATEGORY_INFO if code in unique_cats],
    loc='center left', bbox_to_anchor=(1.02, 0.5),
    fontsize=10, framealpha=0.9,
)
ax.set_axis_off()
plt.tight_layout()

All six planted signals land in their expected categories. Background noise is gray across the board (no pattern detected).

<div class="alert alert-block alert-warning">
<b>Kernel size matters.</b> The kernel determines what spatial scale Gi* looks at. A 3×3 kernel picks up fine-grained clusters; an 11×11 smooths over local variation and only detects broad patterns. If your features are smaller than the kernel, they get averaged out. Use <code>circle_kernel</code> or <code>annulus_kernel</code> from <code>xrspatial.convolution</code> for circular neighborhoods.
</div>

## Mann-Kendall trend map

The `trend_zscore` variable shows how strongly each pixel is trending, separate from the discrete categories. Positive values mean increasing cluster intensity over time. The p-value says whether the trend is statistically significant; the category classifier uses a 5% threshold on this test.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

vlim = float(np.nanpercentile(np.abs(result['trend_zscore'].values), 99))
result['trend_zscore'].plot.imshow(
    ax=ax1, cmap='RdBu_r', vmin=-vlim, vmax=vlim,
    add_colorbar=True, cbar_kwargs={'label': 'Z-score', 'shrink': 0.7})
ax1.set_title('Mann-Kendall trend z-score', fontsize=13)
ax1.set_axis_off()

pval = result['trend_pvalue'].values
tp_log = xr.DataArray(
    -np.log10(np.clip(pval, 1e-20, 1.0)),
    dims=result['trend_pvalue'].dims,
    coords=result['trend_pvalue'].coords,
)
tp_log.plot.imshow(
    ax=ax2, cmap='magma',
    add_colorbar=True, cbar_kwargs={'label': '-log10(p)', 'shrink': 0.7})
ax2.set_title('Trend significance', fontsize=13)
ax2.set_axis_off()

plt.tight_layout()

The intensifying block (top-right) has a strong positive z-score and a tiny p-value: the upward trend is real. The diminishing block is the opposite, with a negative z-score of similar magnitude. Persistent blocks sit near zero because nothing is changing.

## Time series at individual pixels

The `gi_zscore` and `gi_bin` variables give you the full Gi* time series at any pixel. Here we pull out three locations and plot the z-score as bar charts. Red bars are significant hot spots, blue are significant cold, gray is not significant. Dashed lines mark the 95% confidence threshold (z = ±1.96).

In [ ]:
pixels = {
    'Persistent Hot (4)':    (45, 25),
    'Intensifying Hot (3)':  (14, 79),
    'Historical Hot (8)':    (84, 79),
}

fig, axes = plt.subplots(len(pixels), 1, figsize=(12, 3.2 * len(pixels)),
                         sharex=True)
time_steps = np.arange(n_times)

for ax, (label, (py, px)) in zip(axes, pixels.items()):
    zs = result['gi_zscore'].values[:, py, px]
    bins = result['gi_bin'].values[:, py, px]
    cat_code = int(result['category'].values[py, px])

    bar_colors = ['#fb6a4a' if b >= 90 else '#6baed6' if b <= -90 else '#d9d9d9'
                  for b in bins]
    ax.bar(time_steps, zs, color=bar_colors, edgecolor='white', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axhline(1.96, color='gray', linewidth=0.5, linestyle='--', alpha=0.5)
    ax.axhline(-1.96, color='gray', linewidth=0.5, linestyle='--', alpha=0.5)
    ax.set_ylabel('Gi* z-score')
    ax.set_title(f'{label}   (y={py}, x={px},  category={cat_code:+d})',
                 fontsize=12)

axes[-1].set_xlabel('Time step')
axes[-1].set_xticks(time_steps)

fig.legend(
    handles=[Patch(facecolor='#fb6a4a', edgecolor='gray', label='Significant hot'),
             Patch(facecolor='#6baed6', edgecolor='gray', label='Significant cold'),
             Patch(facecolor='#d9d9d9', edgecolor='gray', label='Not significant')],
    loc='lower center', ncol=3, fontsize=10, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()

Persistent: red all the way through, roughly the same height. Intensifying: starts low and grows over time, which is what Mann-Kendall picks up. Historical: red for most of the run, then drops to gray at the end. It was hot but isn't anymore.

## Confidence bins over time

The `gi_bin` variable holds the Gi* confidence level (±90%, ±95%, ±99%, or not significant) at each pixel and time step. Plotting a few steps side by side shows when and where clusters appear or disappear.

In [ ]:
steps_to_show = [0, 6, 13, 19]
bin_cmap = ListedColormap(['#08519c', '#6baed6', '#c6dbef',
                           '#d9d9d9',
                           '#fcbba1', '#fb6a4a', '#cb181d'])
bin_bounds = [-99.5, -95.5, -90.5, -0.5, 0.5, 90.5, 95.5, 99.5]
bin_norm = BoundaryNorm(bin_bounds, bin_cmap.N)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, t in zip(axes, steps_to_show):
    result['gi_bin'].isel(time=t).plot.imshow(
        ax=ax, cmap=bin_cmap, norm=bin_norm, add_colorbar=False,
        interpolation='nearest')
    ax.set_title(f't = {t}', fontsize=12)
    ax.set_axis_off()

fig.legend(
    handles=[Patch(facecolor=c, edgecolor='gray', label=l)
             for c, l in zip(bin_cmap.colors,
                             ['-99%', '-95%', '-90%', 'n.s.',
                              '+90%', '+95%', '+99%'])],
    loc='lower center', ncol=7, fontsize=9, bbox_to_anchor=(0.5, -0.06))
plt.tight_layout()

The new hot spot (top-left) only lights up at t=19. The historical one (bottom-right) is bright through t=13 but gone by the final frame. Persistent blocks don't budge.

<div class="alert alert-block alert-warning">
<b>Edge pixels.</b> The default <code>boundary='nan'</code> sets pixels within the kernel radius of the edge to NaN, so they always classify as "no pattern." If you need classifications at the edges, use <code>boundary='nearest'</code> or <code>'reflect'</code>. These fill in the missing neighbors with assumed values, which trades coverage for statistical rigor.
</div>

### References

- Getis, A. and Ord, J. K. (1992). The analysis of spatial association by use of distance statistics. *Geographical Analysis*, 24(3), 189-206.
- Mann, H. B. (1945). Nonparametric tests against trend. *Econometrica*, 13(3), 245-259.
- Kendall, M. G. (1975). *Rank Correlation Methods.* 4th ed. London: Charles Griffin.
- [Emerging hot spot analysis concepts](https://pro.arcgis.com/en/pro-app/latest/tool-reference/space-time-pattern-mining/emerginghotspots.htm), Esri